#### 5. Explore how the critical Wasserstein radius ε∗ depends on the sample size, k.

In [ ]:
import time

def find_critical_epsilon(eps_grid, performance_values):
    """
    Находит критическое значение ε*, минимизирующее out-of-sample performance.
    
    Parameters:
    -----------
    eps_grid : np.ndarray
        Массив значений ε
    performance_values : np.ndarray
        Массив средних значений out-of-sample performance (ρ_out)
    
    Returns:
    --------
    epsilon_star : float
        Критическое значение ε
    """
    # Находим индекс минимального значения performance
    min_idx = np.argmin(performance_values)
    epsilon_star = eps_grid[min_idx]
    
    # Дополнительно: квадратичная интерполяция для более точного определения
    if 0 < min_idx < len(eps_grid) - 1:
        # Используем параболическую интерполяцию
        x = eps_grid[min_idx-1:min_idx+2]
        y = performance_values[min_idx-1:min_idx+2]
        
        # Коэффициенты квадратичного полинома
        coeffs = np.polyfit(x, y, 2)
        
        # Вершина параболы
        if coeffs[0] != 0:
            epsilon_star_interp = -coeffs[1] / (2 * coeffs[0])
            # Ограничиваем интерполяцию разумными пределами
            if x[0] <= epsilon_star_interp <= x[2]:
                epsilon_star = epsilon_star_interp
    
    return epsilon_star


def explore_epsilon_vs_sample_size(
    env,
    generator,
    n_fixed=10,                    # фиксированный размер задачи
    k_values=[5, 10, 20, 30, 50, 75, 100, 150, 200],# размеры выборки для исследования
    eps_grid=None,                 # сетка ε (если None, будет создана автоматически)
    num_instances=20,              # количество тестовых инстансов для каждого k
    K_test=500,                    # размер тестовой выборки
    alpha=0.95                     # уровень доверия CVaR
):
    """
    Исследует зависимость критического радиуса Вассерштейна ε* от размера выборки k.
    
    Parameters:
    -----------
    env : Gurobi environment
    generator : DRODataGenerator
        Генератор данных
    n_fixed : int
        Фиксированный размер задачи (n)
    k_values : list
        Список размеров выборки для исследования
    eps_grid : np.ndarray or None
        Сетка значений ε
    num_instances : int
        Количество инстансов для усреднения
    K_test : int
        Размер тестовой выборки
    alpha : float
        Уровень доверия CVaR
    
    Returns:
    --------
    results : dict
        Словарь с результатами для каждого k
    """
    
    if eps_grid is None:
        # Адаптивная сетка ε в зависимости от k
        eps_grid = np.array([0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0])
    
    results = {}
    
    print(f"Исследование зависимости ε* от размера выборки k")
    print(f"Фиксированный размер задачи n = {n_fixed}")
    print(f"Количество инстансов для каждого k: {num_instances}")
    
    for k_idx, k_train in enumerate(k_values):
        print(f"\n--- Исследуем k = {k_train} ---")
        
        # Для каждого значения ε собираем out-of-sample performance
        eps_performance = []  # (mean_performance, std_performance)
        
        for eps in eps_grid:
            print(f"  ε = {eps:.1f}", end=" ", flush=True)
            
            rho_instances = []
            solve_times = []
            
            for instance_id in tqdm(range(num_instances), desc=f"    k={k_train}, ε={eps}", leave=False):
                # Генерируем обучающую и тестовую выборки
                train_samples = generator.generate_samples(n_fixed, k_train)
                test_samples = generator.generate_samples(n_fixed, K_test)
                
                # Строим uncertainty set
                D, g = build_box_uncertainty(train_samples)
                
                # Решаем DRO задачу
                start_time = time.time()
                solver = DROAssignment(
                    n=n_fixed,
                    epsilon=eps,
                    alpha=alpha,
                    D=D,
                    g=g,
                    env=env
                )
                x_opt, obj_val = solver.solve(train_samples)
                solve_time = time.time() - start_time
                solve_times.append(solve_time)
                
                # Оцениваем out-of-sample performance
                # Используем t из решения (obj_val) для CVaR
                rho = cvar_loss(x_opt, obj_val, test_samples, alpha=alpha)
                rho_instances.append(rho)
            
            mean_rho = np.mean(rho_instances)
            std_rho = np.std(rho_instances)
            mean_time = np.mean(solve_times)
            eps_performance.append((mean_rho, std_rho, mean_time))
            
            print(f"ρ = {mean_rho:.4f} ± {std_rho:.4f}, время = {mean_time:.2f}s")
        
        # Находим критическое ε* для данного k
        means = [p[0] for p in eps_performance]
        epsilon_star = find_critical_epsilon(eps_grid, np.array(means))
        
        results[k_train] = {
            'eps_grid': eps_grid,
            'performance': eps_performance,
            'epsilon_star': epsilon_star,
            'solve_times': [p[2] for p in eps_performance]
        }
        
        print(f"\n  >>> Для k = {k_train}: ε* ≈ {epsilon_star:.4f}")
    
    return results


def plot_epsilon_star_vs_sample_size(results, n_fixed, theoretical_fit=True):
    """
    Строит график зависимости ε* от размера выборки k.
    
    Parameters:
    -----------
    results : dict
        Результаты из функции explore_epsilon_vs_sample_size
    n_fixed : int
        Размер задачи (для подписи)
    theoretical_fit : bool
        Добавить ли теоретическую аппроксимацию ε* ~ 1/√k
    """
    
    k_values = sorted(results.keys())
    epsilon_stars = [results[k]['epsilon_star'] for k in k_values]
    
    plt.figure(figsize=(10, 6))
    
    # Основной график
    plt.plot(k_values, epsilon_stars, 'bo-', linewidth=2, markersize=8, label='ε* (эксперимент)')
    plt.xlabel('Размер выборки k', fontsize=12)
    plt.ylabel('Критический радиус ε*', fontsize=12)
    plt.title(f'Зависимость ε* от размера выборки (n = {n_fixed})', fontsize=14)
    plt.grid(True, alpha=0.3)
    
    # Теоретическая аппроксимация ε* ~ C/√k
    if theoretical_fit and len(k_values) >= 2:
        # Используем только точки с ε* > 0 для подбора
        valid_idx = [i for i, eps in enumerate(epsilon_stars) if eps > 0]
        if len(valid_idx) >= 2:
            k_valid = [k_values[i] for i in valid_idx]
            eps_valid = [epsilon_stars[i] for i in valid_idx]
            
            # Подбираем константу C
            C = np.mean([eps * np.sqrt(k) for eps, k in zip(eps_valid, k_valid)])
            k_theor = np.linspace(min(k_values), max(k_values), 100)
            eps_theor = C / np.sqrt(k_theor)
            
            plt.plot(k_theor, eps_theor, 'r--', linewidth=2, 
                    label=f'Теоретическая: ε* = {C:.2f}/√k')
    
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_epsilon_star_convergence(results, n_fixed):
    """
    Строит график сходимости ε* с ростом выборки.
    """
    k_values = sorted(results.keys())
    epsilon_stars = [results[k]['epsilon_star'] for k in k_values]
    
    plt.figure(figsize=(10, 6))
    
    # Основной график
    plt.plot(k_values, epsilon_stars, 'gs-', linewidth=2, markersize=8, label='ε*')
    
    # Горизонтальная линия на уровне 0
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='Асимптота (ε*=0)')
    
    plt.xlabel('Размер выборки k', fontsize=12)
    plt.ylabel('Критический радиус ε*', fontsize=12)
    plt.title(f'Сходимость ε* с ростом выборки (n = {n_fixed})', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


# Запуск исследования
def run_task_5():
    """
    Запуск полного исследования для пункта 5.
    """
    # Инициализация окружения Gurobi
    env = gp.Env(empty=True)
    set_environ(env)
    env.start()
    
    # Создание генератора данных
    generator = DRODataGenerator(seed=42)
    
    # Параметры исследования
    n_fixed = 10  # фиксированный размер задачи
    
    # Размеры выборки для исследования
    k_values = [5, 10, 15, 20, 25, 30, 40, 50]
    
    # Сетка значений ε
    eps_grid = np.array([0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0])
    
    # Количество инстансов
    num_instances = 10
    K_test = 300
    
    print("ПУНКТ 5: Исследование зависимости критического радиуса ε* от размера выборки k")
    print(f"Параметры:")
    print(f"  - Размер задачи n = {n_fixed}")
    print(f"  - Исследуемые k: {k_values}")
    print(f"  - Сетка ε: {eps_grid}")
    print(f"  - Количество инстансов: {num_instances}")
    print(f"  - Размер тестовой выборки: {K_test}")
    
    # Запуск исследования
    results = explore_epsilon_vs_sample_size(
        env=env,
        generator=generator,
        n_fixed=n_fixed,
        k_values=k_values,
        eps_grid=eps_grid,
        num_instances=num_instances,
        K_test=K_test,
        alpha=0.95
    )
    
    # Закрываем окружение
    env.close()
    
    # Визуализация результатов
    plot_epsilon_star_vs_sample_size(results, n_fixed, theoretical_fit=True)
    plot_epsilon_star_convergence(results, n_fixed)
    
    # Вывод таблицы результатов
    print("\nСводная таблица результатов:")
    print(f"{'k':<10} {'ε*':<15} {'Минимальное ρ':<20}")
    for k in sorted(results.keys()):
        eps_star = results[k]['epsilon_star']
        min_rho = min([p[0] for p in results[k]['performance']])
        print(f"{k:<10} {eps_star:<15.6f} {min_rho:<20.6f}")
    
    return results


# Пример запуска
results = run_task_5()

# Дополнительная функция: сравнение ε* для разных n
def compare_epsilon_star_for_different_n(
    env,
    generator,
    n_values=[5, 10, 15, 20],
    k_values=[10, 20, 30, 50, 100],
    eps_grid=None,
    num_instances=10
):
    """
    Сравнение зависимости ε*(k) для разных размеров задачи n.
    """
    if eps_grid is None:
        eps_grid = np.array([0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0])
    
    results_by_n = {}
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(n_values)))
    
    for n, color in zip(n_values, colors):
        print(f"\n=== Исследование для n = {n} ===")
        
        results = explore_epsilon_vs_sample_size(
            env=env,
            generator=generator,
            n_fixed=n,
            k_values=k_values,
            eps_grid=eps_grid,
            num_instances=num_instances,
            K_test=300,
            alpha=0.95
        )
        
        results_by_n[n] = results
        
        # Строим линию для данного n
        k_list = sorted(results.keys())
        eps_star_list = [results[k]['epsilon_star'] for k in k_list]
        plt.plot(k_list, eps_star_list, 'o-', linewidth=2, markersize=6, 
                color=color, label=f'n = {n}')
    
    plt.xlabel('Размер выборки k', fontsize=12)
    plt.ylabel('Критический радиус ε*', fontsize=12)
    plt.title('Зависимость ε* от k для разных размеров задачи n', fontsize=14)
    plt.xscale('log')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    return results_by_n

# Для сравнения разных n (более ресурсоемко)
# results_by_n = compare_epsilon_star_for_different_n(env, generator)